# DACON ETRI post-ID335 sequence source report-only runner

This Kaggle runner executes a memory-safe S1 raw-sequence random-convolution refresh after the ID335 workflow review. It is **report-only**: it does not create DACON submissions, promoted notebooks, base predictions, or submission-log rows. The purpose is to collect small diagnostic JSON files from heavier raw-sequence jobs while keeping the local machine out of the expensive loop.

## 1. Locate the packaged project payload

The runner accepts either a `source.zip` from the Kaggle dataset package or an unpacked project tree under `/kaggle/input`. It then works from an isolated temporary project directory and stages small outputs under `/kaggle/working/kaggle_outputs`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def run(cmd: str) -> None:
    print(f"\n$ {cmd}", flush=True)
    env = os.environ.copy()
    env.setdefault("PYTHONHASHSEED", "0")
    env.setdefault("OMP_NUM_THREADS", "2")
    env.setdefault("OPENBLAS_NUM_THREADS", "2")
    env.setdefault("MKL_NUM_THREADS", "2")
    env.setdefault("NUMEXPR_NUM_THREADS", "2")
    subprocess.run(cmd, shell=True, check=True, env=env)


is_kaggle = Path("/kaggle/input").exists()
work_dir = Path("/kaggle/working") if is_kaggle else Path.cwd()
project_root = Path("/tmp/dacon_project_sequence_source_post_id335") if is_kaggle else Path.cwd()
output_root = work_dir / "kaggle_outputs" if is_kaggle else Path("kaggle_outputs")
output_root.mkdir(parents=True, exist_ok=True)
os.environ["KAGGLE_OUTPUT_DIR"] = str(output_root)

for key, value in {
    "PYTHONHASHSEED": "0",
    "OMP_NUM_THREADS": "2",
    "OPENBLAS_NUM_THREADS": "2",
    "MKL_NUM_THREADS": "2",
    "NUMEXPR_NUM_THREADS": "2",
}.items():
    os.environ.setdefault(key, value)

required_payload_paths = [
    Path("scripts/diagnose_public_random_conv_sequence.py"),
    Path("artifacts/base_predictions/base_oof_id_173_id169_s4_hourly_sequence_overlay_cv0.545253.csv"),
    Path("artifacts/base_predictions/base_test_id_173_id169_s4_hourly_sequence_overlay_cv0.545253.csv"),
    Path("data/raw/ch2025_data_items"),
]


def payload_score(root: Path) -> int:
    return sum((root / rel).exists() for rel in required_payload_paths)


if is_kaggle:
    print("Detected Kaggle environment; locating project payload...")
    source_zips = sorted(Path("/kaggle/input").glob("**/source.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    project_root.mkdir(parents=True, exist_ok=True)
    if source_zips:
        with zipfile.ZipFile(source_zips[0]) as zf:
            zf.extractall(project_root)
        print(f"Extracted {source_zips[0]} -> {project_root}")
    else:
        payload_roots = []
        for root, dirs, _files in os.walk("/kaggle/input"):
            if {"src", "conf", "scripts"}.issubset(set(dirs)):
                payload_roots.append(Path(root))
        if not payload_roots:
            raise FileNotFoundError("No source.zip or unpacked project root with src/conf/scripts found in Kaggle inputs.")
        scored = [(payload_score(root), root) for root in payload_roots]
        scored.sort(reverse=True)
        best_score, best_root = scored[0]
        if best_score == 0:
            raise FileNotFoundError("Found project-like roots, but none contain required sequence-source payload files.")
        if project_root.exists():
            shutil.rmtree(project_root)
        shutil.copytree(best_root, project_root)
        print(f"Copied unpacked payload {best_root} -> {project_root}; score={best_score}")
else:
    print("Local environment detected; using current repository root.")

os.chdir(project_root)
print("project_root =", project_root)
print("output_root =", output_root)
missing_required = [str(rel) for rel in required_payload_paths if not (project_root / rel).exists()]
if missing_required:
    raise FileNotFoundError(f"Missing required payload paths: {missing_required}")
print("Required payload paths verified.")

## 2. Ensure Kaggle runtime dependencies only when missing

Kaggle usually has most packages preinstalled. To keep the runner polite and reproducible, this cell checks imports first and installs only absent packages.

In [ ]:
if is_kaggle:
    import importlib.util

    required = {
        "lightgbm": "lightgbm",
        "xgboost": "xgboost",
        "catboost": "catboost",
        "nbformat": "nbformat",
        "holidays": "holidays",
        "pyarrow": "pyarrow",
        "fastparquet": "fastparquet",
        "sklearn": "scikit-learn",
    }
    missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
    print("missing packages =", missing)
    if missing:
        run(f"{sys.executable} -m pip install -q {' '.join(missing)}")
else:
    print("Skipping dependency installation outside Kaggle.")

## 3. Rebuild processed caches if the packaged cache is absent

The random-conv diagnostic ultimately uses raw sensor parquet files, but several project helpers expect processed day-level caches. This cell rebuilds those caches only when needed.

In [ ]:
from pathlib import Path

import pandas as pd

train_cache = Path("data/processed/df_train_raw.parquet")
test_cache = Path("data/processed/df_test_raw.parquet")
if train_cache.exists() and test_cache.exists():
    print("Processed caches already exist; skipping rebuild.")
else:
    from src.data.make_dataset import build_training_dataset

    raw_dir = Path("data/raw")
    sensor_dir = raw_dir / "ch2025_data_items"
    if not sensor_dir.exists():
        raise FileNotFoundError(f"Missing raw sensor dir: {sensor_dir}")
    sensor_data = {}
    for parquet_path in sorted(sensor_dir.glob("*.parquet")):
        print("Loading", parquet_path)
        sensor_data[parquet_path.stem] = pd.read_parquet(parquet_path)
    metrics = pd.read_csv(raw_dir / "ch2026_metrics_train.csv")
    sample = pd.read_csv(raw_dir / "ch2026_submission_sample.csv")
    train_df = build_training_dataset(metrics, sensor_data)
    test_df = build_training_dataset(sample, sensor_data)
    train_cache.parent.mkdir(parents=True, exist_ok=True)
    train_df.to_parquet(train_cache, index=False)
    test_df.to_parquet(test_cache, index=False)
    print("Wrote processed caches:", train_df.shape, test_df.shape)

## 4. Run post-ID335 S1 random-conv report-only jobs

These jobs intentionally start from ID173, the pre-random-conv S4 hourly-sequence anchor, so the sweep can test whether a fresh S1 sequence representation has independent signal. They run sequentially to cap memory. If a JSON file already exists, the job is skipped.

In [ ]:
# Report-only post-ID335 S1 random-conv source refresh. No promote flag, no submission artifacts.
import gc
import time
from pathlib import Path

RUN_STARTED_TS = time.time()
SWEEP_JOBS = [
    {"base_id": 173, "n_kernels": 512, "seed": 45501, "output": "reports/id173_random_conv_sequence_post_id335_s1_nk512_seed45501_20260617.json"},
    {"base_id": 173, "n_kernels": 512, "seed": 45502, "output": "reports/id173_random_conv_sequence_post_id335_s1_nk512_seed45502_20260617.json"},
    {"base_id": 173, "n_kernels": 768, "seed": 45601, "output": "reports/id173_random_conv_sequence_post_id335_s1_nk768_seed45601_20260617.json"},
    {"base_id": 173, "n_kernels": 768, "seed": 45602, "output": "reports/id173_random_conv_sequence_post_id335_s1_nk768_seed45602_20260617.json"},
    {"base_id": 173, "n_kernels": 1024, "seed": 45701, "output": "reports/id173_random_conv_sequence_post_id335_s1_nk1024_seed45701_20260617.json"},
    {"base_id": 173, "n_kernels": 1024, "seed": 45702, "output": "reports/id173_random_conv_sequence_post_id335_s1_nk1024_seed45702_20260617.json"},
]
STRICT_EXPERIMENT_COMMAND = (
    f"post-ID335 S1 random-conv sequence source report-only sweep: {len(SWEEP_JOBS)} jobs; "
    f"base_ids={sorted({job['base_id'] for job in SWEEP_JOBS})}; "
    f"kernels={sorted({job['n_kernels'] for job in SWEEP_JOBS})}; "
    f"seeds={[job['seed'] for job in SWEEP_JOBS]}"
)
print(STRICT_EXPERIMENT_COMMAND)
for job in SWEEP_JOBS:
    output_path = Path(job["output"])
    if output_path.exists():
        print(f"Skipping existing output: {output_path}")
        continue
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = (
        f"{sys.executable} scripts/diagnose_public_random_conv_sequence.py "
        f"--base-id {job['base_id']} "
        f"--n-kernels {job['n_kernels']} "
        f"--seed {job['seed']} "
        f"--output {output_path}"
    )
    run(cmd)
    gc.collect()
print("Finished report-only sequence source jobs.")

## 5. Stage only small diagnostic outputs

Only the post-ID335 S1 JSON reports and a manifest are copied to Kaggle outputs. This runner deliberately avoids staging `submissions/`, promoted notebooks, base predictions, or submission logs.

In [ ]:
# Stage small report-only artifacts for `kaggle kernels output`.
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

out_dir = Path(os.environ.get("KAGGLE_OUTPUT_DIR", "kaggle_outputs"))
out_dir.mkdir(parents=True, exist_ok=True)
patterns = [
    "reports/id173_random_conv_sequence_post_id335_s1_*.json",
]
copied = []
run_started_ts = float(globals().get("RUN_STARTED_TS", 0.0))
for pattern in patterns:
    for path in sorted(Path(".").glob(pattern), key=lambda p: p.stat().st_mtime):
        if not path.is_file():
            continue
        if path.stat().st_mtime < run_started_ts - 5.0:
            continue
        dest = out_dir / path.name
        shutil.copy2(path, dest)
        copied.append(str(path))
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "command": STRICT_EXPERIMENT_COMMAND,
    "sweep_jobs": globals().get("SWEEP_JOBS", []),
    "copied_artifacts": copied,
    "note": (
        "Report-only post-ID335 sequence source sweep. Pull these JSON files locally and review diagnostics before any "
        "promotion. No DACON submission, submission-log row, base prediction, or notebook/history artifact is created here."
    ),
}
(out_dir / "remote_sequence_source_post_id335_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(manifest, ensure_ascii=False, indent=2))